# A New Branch-and-Bound Pruning Framework for L0-Regularized Problems

*T. Guyard, C. Elvira, C. Herzet, and A.-N. Arslan. ICML 2024.*

#### Notebook to reproduce experiments of Section 4.1

---

These experiments compare the performance of different methods to solver L0-norm problem associated with the reconstruction of sparse signals. To reproduce it, let first import all the necessary packages and routines.

In [ ]:
import numpy as np
from el0ps.utils import compute_lmbd_max
from l0exp.calibration import get_calibration_l0learn
from l0exp.dataset import get_dataset_synthetic
from l0exp.solver import get_solver, can_handle_instance

Now, let generate the data, that is, a noisy signal `y = Ax + Ɛ`, where:

- `A in R^{m x n}` is a matrix with rows drawn from a multivariate Normal distribution `N(0,K)` with covariance matrix `K in R^{n x n}` defined as `K[i,j] = r^|i-j|`,
- `x in R^{n}` is a sparse signal with `k` non-zero entries evenly spaces with unit amplitude,
- `Ɛ in R^{m}` is a Gaussian noise with signal-to-noise ratio `s` with respect to `Ax` on average.

In [ ]:
synthetic_args = {
    'k'             : 5,        # number of non-zero entries in x
    'm'             : 500,      # number of rows of A / entries in y
    'n'             : 1000,     # number of columns of A / entries in x
    'r'             : 0.9,      # correlation matrix coefficient
    's'             : 10.0,     # signal-to-noise ratio
    'seed'          : None,     # optional int seed for reproducibility (no seed set if None)
}


A, y, x = get_dataset_synthetic(**synthetic_args)
print(f"A shape: {A.shape}")
print(f"y shape: {y.shape}")
print(f"x shape: {x.shape}")
print(f"nnz    : {np.count_nonzero(x)}")
print(f"snr    : {np.linalg.norm(A @ x)**2 / np.linalg.norm(y - A @ x)**2:.2f}")

We now use the `L0Learn` package to calibrate appropriate data-fidelity and penalty functions, as well as the value of the regularization parameter `lmbd`. In our experiments, we used a leastsquares data-fidelity and a big-M penalty.

In [ ]:
datafit, penalty, lmbd = get_calibration_l0learn(A, y, x, "Leastsquares", "Bigm")
print(f"datafit   : {datafit}")
print(f"penalty   : {penalty}")
print(f"lambda    : {lmbd}")
print(f"lambda_max: {compute_lmbd_max(datafit, penalty, A)}")

Finally, we can define the different methods to compare and their parameters. Here is the setup used in our experiments.

- For the **Performance profile**, this solving procedure has been repeated `100` times to plot performance profiles with the number of instances solved within a given time limit.
- For the **Sensibility study**, this solving procedure has been performed with different parameters in `synthetic_args` and repeated `10` times to perform the sensitivity analysis.

Solvers that cannot handle the considered instance are automatically skipped.

In [ ]:
# Solvers to use (comment out those you don't want to run or that are not installed)
solver_types = [
    "el0ps",
    "l0bnb",
    "mimosa",
    "gurobi",
    "mosek",
    "oa"
]

solver_args = {
    "time_limit"    : 3600,   # time limit in seconds for solvers
    "relative_gap"  : 1.e-8,  # relative optimality gap on the objective value
    "verbose"       : False,  # verbosity toggle for solvers
}


for solver_type in solver_types:

    try:    
        solver = get_solver(solver_type, solver_args)
        if can_handle_instance(solver, datafit, penalty):
            print(f"Running {solver_type}...")
            result = solver.solve(datafit, penalty, A, lmbd)
            print(f"  status    : {result.status}")
            print(f"  objective : {result.objective_value:.4f}")
            print(f"  non-zeros : {np.count_nonzero(result.x)}")
            print(f"  solve time: {result.solve_time:.4f}")
        else:
            print(f"Skipping {solver_type}: cannot handle this instance")
    except Exception as e:
        print(f"Solver {solver_type} failed with error: {e}")
    print()